In [1]:
# ============================================================
# Cell 1: Environment + Project Path
# ============================================================

from pathlib import Path
import sys
import os
import torch


# ------------------------------------------------------------
# Find MyGPT2 project root
# ------------------------------------------------------------

def find_project_root():
    """
    Locate the MyGPT2 project root by looking for
    model/, tokenizer/, training/ and train.py.
    """

    candidates = []

    # Current working directory
    cwd = Path.cwd().resolve()
    candidates.append(cwd)

    # Walk upward from current directory
    candidates.extend(cwd.parents)

    # Notebook file location, if available
    try:
        notebook_dir = Path(
            os.environ.get(
                "PWD",
                str(cwd)
            )
        ).resolve()

        candidates.append(notebook_dir)
        candidates.extend(notebook_dir.parents)

    except Exception:
        pass

    # Look for project signature
    for candidate in candidates:

        if (
            (candidate / "model").is_dir()
            and
            (candidate / "tokenizer").is_dir()
            and
            (candidate / "training").is_dir()
            and
            (candidate / "train.py").is_file()
        ):
            return candidate

    return None


PROJECT_ROOT = find_project_root()


if PROJECT_ROOT is None:

    raise RuntimeError(
        "Could not automatically find the MyGPT2 project root.\n\n"
        "Expected structure:\n"
        "D:\\Gpt2_v01\\\n"
        "├── model\\\n"
        "├── tokenizer\\\n"
        "├── training\\\n"
        "└── train.py\n\n"
        f"Current working directory:\n{Path.cwd()}"
    )


# ------------------------------------------------------------
# Add project root to Python path
# ------------------------------------------------------------

PROJECT_ROOT = PROJECT_ROOT.resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# ------------------------------------------------------------
# Verify project directories
# ------------------------------------------------------------

required_directories = [
    PROJECT_ROOT / "model",
    PROJECT_ROOT / "tokenizer",
    PROJECT_ROOT / "training",
]

for directory in required_directories:

    if not directory.is_dir():

        raise RuntimeError(
            f"Required project directory missing:\n"
            f"{directory}"
        )


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

if torch.cuda.is_available():

    DEVICE = torch.device("cuda")

else:

    DEVICE = torch.device("cpu")


# ------------------------------------------------------------
# Display environment
# ------------------------------------------------------------

print("=" * 75)
print("MyGPT2 Validation Evaluation")
print("=" * 75)

print(
    f"Current Working Directory : "
    f"{Path.cwd()}"
)

print(
    f"Project Root              : "
    f"{PROJECT_ROOT}"
)

print(
    f"Python                    : "
    f"{sys.version.split()[0]}"
)

print(
    f"PyTorch                   : "
    f"{torch.__version__}"
)

print(
    f"Device                    : "
    f"{DEVICE}"
)

if torch.cuda.is_available():

    print(
        f"GPU                       : "
        f"{torch.cuda.get_device_name(0)}"
    )

print("=" * 75)

print()
print("Project structure : ✅ FOUND")
print("Python path       : ✅ CONFIGURED")

MyGPT2 Validation Evaluation
Current Working Directory : d:\Gpt2_v01\evaluation\notebooks
Project Root              : D:\Gpt2_v01
Python                    : 3.14.3
PyTorch                   : 2.13.0+cu132
Device                    : cuda
GPU                       : NVIDIA GeForce RTX 5060 Ti

Project structure : ✅ FOUND
Python path       : ✅ CONFIGURED


In [2]:
# ============================================================
# Cell 2: MyGPT2 Project Imports
# ============================================================

print("=" * 75)
print("Testing MyGPT2 Imports")
print("=" * 75)

from model.config import GPTConfig
from model.model import MyGPTModel
from tokenizer.my_tokenizer import MyGPTTokenizer
from training.checkpoint import load_checkpoint

print()
print("GPTConfig          : ✅ IMPORTED")
print("MyGPTModel         : ✅ IMPORTED")
print("MyGPTTokenizer     : ✅ IMPORTED")
print("load_checkpoint    : ✅ IMPORTED")

print()
print("Project imports    : ✅ PASSED")
print("=" * 75)

Testing MyGPT2 Imports

GPTConfig          : ✅ IMPORTED
MyGPTModel         : ✅ IMPORTED
MyGPTTokenizer     : ✅ IMPORTED
load_checkpoint    : ✅ IMPORTED

Project imports    : ✅ PASSED


In [3]:
# ============================================================
# Cell 3: Paths
# ============================================================

CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "checkpoints"
    / "final_step_00020000.pt"
)

TOKENIZER_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tokenizer"
    / "tokenizer.json"
)

print("=" * 75)
print("Evaluation Paths")
print("=" * 75)

print(f"Checkpoint : {CHECKPOINT_PATH}")
print(f"Tokenizer  : {TOKENIZER_PATH}")

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"\nCheckpoint not found:\n{CHECKPOINT_PATH}"
    )

if not TOKENIZER_PATH.exists():
    raise FileNotFoundError(
        f"\nTokenizer not found:\n{TOKENIZER_PATH}"
    )

print()
print("Checkpoint : ✅ FOUND")
print("Tokenizer  : ✅ FOUND")

Evaluation Paths
Checkpoint : D:\Gpt2_v01\artifacts\checkpoints\final_step_00020000.pt
Tokenizer  : D:\Gpt2_v01\artifacts\tokenizer\tokenizer.json

Checkpoint : ✅ FOUND
Tokenizer  : ✅ FOUND


In [4]:
# ============================================================
# Cell 3: Paths
# ============================================================

CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "checkpoints"
    / "final_step_00020000.pt"
)

TOKENIZER_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tokenizer"
    / "tokenizer.json"
)

print("=" * 75)
print("Evaluation Paths")
print("=" * 75)

print(f"Checkpoint : {CHECKPOINT_PATH}")
print(f"Tokenizer  : {TOKENIZER_PATH}")

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"\nCheckpoint not found:\n{CHECKPOINT_PATH}"
    )

if not TOKENIZER_PATH.exists():
    raise FileNotFoundError(
        f"\nTokenizer not found:\n{TOKENIZER_PATH}"
    )

print()
print("Checkpoint : ✅ FOUND")
print("Tokenizer  : ✅ FOUND")

Evaluation Paths
Checkpoint : D:\Gpt2_v01\artifacts\checkpoints\final_step_00020000.pt
Tokenizer  : D:\Gpt2_v01\artifacts\tokenizer\tokenizer.json

Checkpoint : ✅ FOUND
Tokenizer  : ✅ FOUND


In [5]:
# ============================================================
# Cell 4: Model Configuration
# ============================================================

config = GPTConfig()


# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------

def find_config_value(config, names, default=None):
    """
    Safely find the first existing configuration attribute.
    """
    for name in names:
        if hasattr(config, name):
            value = getattr(config, name)

            if value is not None:
                return value

    return default


# ------------------------------------------------------------
# Read values that actually exist in GPTConfig
# ------------------------------------------------------------

VOCAB_SIZE = find_config_value(
    config,
    ["vocab_size"],
)

HIDDEN_SIZE = find_config_value(
    config,
    ["hidden_size", "n_embd", "d_model"],
)

NUM_LAYERS = find_config_value(
    config,
    ["num_layers", "n_layer", "num_hidden_layers"],
)

NUM_HEADS = find_config_value(
    config,
    [
        "num_heads",
        "num_attention_heads",
        "n_head",
    ],
)

INTERMEDIATE_SIZE = find_config_value(
    config,
    [
        "intermediate_size",
        "ffn_size",
        "n_inner",
    ],
)


# ============================================================
# IMPORTANT
# ============================================================
# GPTConfig does NOT expose sequence length as an attribute.
#
# The actual MyGPT2 training configuration is known from the
# training pipeline:
#
#     Sequence Length = 512
#
# Therefore we explicitly define it here instead of trying
# to guess a nonexistent GPTConfig attribute.
# ============================================================

SEQUENCE_LENGTH = 512


# ------------------------------------------------------------
# Expected training configuration
# ------------------------------------------------------------

EXPECTED_VOCAB_SIZE = 32_000
EXPECTED_SEQUENCE_LENGTH = 512
EXPECTED_HIDDEN_SIZE = 768
EXPECTED_NUM_LAYERS = 12
EXPECTED_NUM_HEADS = 12
EXPECTED_INTERMEDIATE_SIZE = 3072


# ------------------------------------------------------------
# Required configuration checks
# ------------------------------------------------------------

if VOCAB_SIZE is None:
    raise RuntimeError(
        "Could not determine vocabulary size from GPTConfig."
    )

if HIDDEN_SIZE is None:
    raise RuntimeError(
        "Could not determine hidden size from GPTConfig."
    )

if NUM_LAYERS is None:
    raise RuntimeError(
        "Could not determine number of transformer layers "
        "from GPTConfig."
    )

if NUM_HEADS is None:
    raise RuntimeError(
        "Could not determine number of attention heads "
        "from GPTConfig."
    )

if INTERMEDIATE_SIZE is None:
    raise RuntimeError(
        "Could not determine intermediate size "
        "from GPTConfig."
    )


# ============================================================
# Display Configuration
# ============================================================

print("=" * 75)
print("Model Configuration")
print("=" * 75)

print(
    f"Vocabulary Size      : {VOCAB_SIZE:,}"
)

print(
    f"Sequence Length      : {SEQUENCE_LENGTH}"
)

print(
    f"Hidden Size          : {HIDDEN_SIZE}"
)

print(
    f"Transformer Layers   : {NUM_LAYERS}"
)

print(
    f"Attention Heads      : {NUM_HEADS}"
)

print(
    f"Intermediate Size    : {INTERMEDIATE_SIZE}"
)

print("=" * 75)


# ============================================================
# Configuration Validation
# ============================================================

checks = {
    "Vocabulary Size": (
        VOCAB_SIZE,
        EXPECTED_VOCAB_SIZE,
    ),

    "Sequence Length": (
        SEQUENCE_LENGTH,
        EXPECTED_SEQUENCE_LENGTH,
    ),

    "Hidden Size": (
        HIDDEN_SIZE,
        EXPECTED_HIDDEN_SIZE,
    ),

    "Transformer Layers": (
        NUM_LAYERS,
        EXPECTED_NUM_LAYERS,
    ),

    "Attention Heads": (
        NUM_HEADS,
        EXPECTED_NUM_HEADS,
    ),

    "Intermediate Size": (
        INTERMEDIATE_SIZE,
        EXPECTED_INTERMEDIATE_SIZE,
    ),
}


print()
print("=" * 75)
print("Expected Training Configuration Check")
print("=" * 75)

all_config_ok = True

for name, (actual, expected) in checks.items():

    passed = actual == expected

    if not passed:
        all_config_ok = False

    status = "✅" if passed else "❌"

    print(
        f"{name:<25}: "
        f"{status} {actual} "
        f"(expected {expected})"
    )


if not all_config_ok:

    raise RuntimeError(
        "Model configuration does not match "
        "the configuration used during training."
    )


print()
print("Configuration Check : ✅ PASSED")

Model Configuration
Vocabulary Size      : 32,000
Sequence Length      : 512
Hidden Size          : 768
Transformer Layers   : 12
Attention Heads      : 12
Intermediate Size    : 3072

Expected Training Configuration Check
Vocabulary Size          : ✅ 32000 (expected 32000)
Sequence Length          : ✅ 512 (expected 512)
Hidden Size              : ✅ 768 (expected 768)
Transformer Layers       : ✅ 12 (expected 12)
Attention Heads          : ✅ 12 (expected 12)
Intermediate Size        : ✅ 3072 (expected 3072)

Configuration Check : ✅ PASSED


In [6]:
# ============================================================
# Cell 5: Load Tokenizer
# ============================================================

print("=" * 75)
print("Loading Tokenizer")
print("=" * 75)

tokenizer = MyGPTTokenizer.load(
    TOKENIZER_PATH
)

tokenizer_vocab_size = tokenizer.vocabulary_size

print(
    f"Tokenizer Vocabulary : "
    f"{tokenizer_vocab_size:,}"
)

print(
    f"Model Vocabulary     : "
    f"{VOCAB_SIZE:,}"
)

if tokenizer_vocab_size != VOCAB_SIZE:
    raise RuntimeError(
        "Tokenizer/model vocabulary mismatch.\n"
        f"Tokenizer : {tokenizer_vocab_size}\n"
        f"Model     : {VOCAB_SIZE}"
    )

print()
print("Tokenizer : ✅ LOADED")
print("Vocabulary : ✅ MATCHED")

Loading Tokenizer
Tokenizer Vocabulary : 32,000
Model Vocabulary     : 32,000

Tokenizer : ✅ LOADED
Vocabulary : ✅ MATCHED


In [7]:
# ============================================================
# Cell 6: Load Model and Checkpoint
# ============================================================

print("=" * 75)
print("Loading Model")
print("=" * 75)

model = MyGPTModel(
    config
).to(DEVICE)

model.eval()

print("Model created : ✅")


checkpoint = load_checkpoint(
    path=CHECKPOINT_PATH,
    model=model,
    optimizer=None,
    scheduler=None,
    device=DEVICE,
    restore_rng=False,
)


checkpoint_step = checkpoint.get(
    "global_step",
    0,
)

checkpoint_epoch = checkpoint.get(
    "epoch",
    0,
)

training_loss = checkpoint.get(
    "train_loss"
)

validation_loss_saved = checkpoint.get(
    "val_loss"
)


print(
    f"Checkpoint version   : "
    f"{checkpoint.get('checkpoint_version')}"
)

print(
    f"Saved epoch          : "
    f"{checkpoint_epoch}"
)

print(
    f"Saved global step    : "
    f"{checkpoint_step:,}"
)

print(
    f"Saved training loss  : "
    f"{training_loss}"
)

print(
    f"Saved validation loss: "
    f"{validation_loss_saved}"
)

print()
print("Model checkpoint : ✅ LOADED")

Loading Model
Model created : ✅
Checkpoint version   : 1.2
Saved epoch          : 0
Saved global step    : 20,000
Saved training loss  : 5.986759185791016
Saved validation loss: None

Model checkpoint : ✅ LOADED


In [8]:
# ============================================================
# Cell 7: Parameter Verification
# ============================================================

total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


EXPECTED_PARAMETERS = 110_025_216


print("=" * 75)
print("Model Parameters")
print("=" * 75)

print(
    f"Total Parameters     : "
    f"{total_parameters:,}"
)

print(
    f"Total Parameters     : "
    f"{total_parameters / 1_000_000:.2f}M"
)

print(
    f"Trainable Parameters : "
    f"{trainable_parameters:,}"
)


if total_parameters != EXPECTED_PARAMETERS:
    raise RuntimeError(
        "Parameter count mismatch.\n"
        f"Expected : {EXPECTED_PARAMETERS:,}\n"
        f"Actual   : {total_parameters:,}"
    )

print()
print("Parameter Count : ✅ MATCH")

Model Parameters
Total Parameters     : 110,025,216
Total Parameters     : 110.03M
Trainable Parameters : 110,025,216

Parameter Count : ✅ MATCH


In [9]:
# ============================================================
# Cell 8: Load TinyStories Validation Dataset
# ============================================================

from datasets import load_dataset


print("=" * 75)
print("Loading TinyStories Validation Dataset")
print("=" * 75)

print("Dataset : roneneldan/TinyStories")
print("Split   : validation")

validation_dataset = load_dataset(
    "roneneldan/TinyStories",
    split="validation",
)

print()
print(
    f"Validation documents : "
    f"{len(validation_dataset):,}"
)

print("Dataset loading      : ✅ PASSED")

d:\Gpt2_v01\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading TinyStories Validation Dataset
Dataset : roneneldan/TinyStories
Split   : validation



Validation documents : 21,990
Dataset loading      : ✅ PASSED


In [10]:
# ============================================================
# Cell 9: Dataset Inspection
# ============================================================

print("=" * 75)
print("Validation Dataset Inspection")
print("=" * 75)

print(
    "Columns:"
)

print(
    validation_dataset.column_names
)

if len(validation_dataset) > 0:

    first_sample = validation_dataset[0]

    print()
    print(
        "First sample keys:"
    )

    print(
        first_sample.keys()
    )

    print()
    print(
        "First sample preview:"
    )

    print(
        first_sample["text"][:500]
    )

print()
print("Dataset inspection : ✅ PASSED")

Validation Dataset Inspection
Columns:
['text']

First sample keys:
dict_keys(['text'])

First sample preview:
Spot. Spot saw the shiny car and said, "Wow, Kitty, your car is so bright and clean!" Kitty smiled and replied, "Thank you, Spot. I polish it every day."

After playing with the car, Kitty and Spot felt thirsty. They found a small pond with clear water. They drank the water and felt very happy. They played together all day and became best friends.

Dataset inspection : ✅ PASSED


In [11]:
# ============================================================
# Cell 10: Tokenizer Test
# ============================================================

sample_text = validation_dataset[0]["text"]

encoded = tokenizer.encode(
    sample_text
)

if hasattr(encoded, "ids"):
    token_ids = encoded.ids
else:
    token_ids = encoded


print("=" * 75)
print("Tokenizer Test")
print("=" * 75)

print(
    f"Characters : {len(sample_text)}"
)

print(
    f"Tokens     : {len(token_ids)}"
)

print(
    f"First tokens: {token_ids[:20]}"
)


if len(token_ids) == 0:
    raise RuntimeError(
        "Tokenizer produced zero tokens."
    )

print()
print("Tokenizer test : ✅ PASSED")

Tokenizer Test
Characters : 349
Tokens     : 84
First tokens: [2, 2753, 17, 2753, 609, 221, 2298, 728, 239, 350, 15, 333, 3447, 15, 10346, 15, 539, 728, 298, 371]

Tokenizer test : ✅ PASSED


In [12]:
# ============================================================
# Cell 11: Build Validation Token Stream
# ============================================================

print("=" * 75)
print("Building Validation Token Stream")
print("=" * 75)

MAX_EVALUATION_TOKENS = 500_000

all_tokens = []

documents_processed = 0
documents_skipped = 0


for document in validation_dataset:

    text = document.get("text", "")

    if not isinstance(text, str):
        documents_skipped += 1
        continue

    text = text.strip()

    if not text:
        documents_skipped += 1
        continue

    encoded = tokenizer.encode(text)

    if hasattr(encoded, "ids"):
        ids = encoded.ids
    else:
        ids = encoded

    if not ids:
        documents_skipped += 1
        continue

    all_tokens.extend(
        int(token_id)
        for token_id in ids
    )

    documents_processed += 1

    if len(all_tokens) >= MAX_EVALUATION_TOKENS:
        break


all_tokens = all_tokens[
    :MAX_EVALUATION_TOKENS
]


print(
    f"Documents processed : "
    f"{documents_processed:,}"
)

print(
    f"Documents skipped   : "
    f"{documents_skipped:,}"
)

print(
    f"Tokens collected    : "
    f"{len(all_tokens):,}"
)


if len(all_tokens) < SEQUENCE_LENGTH + 1:
    raise RuntimeError(
        "Not enough validation tokens to create "
        "a single validation sequence."
    )

print()
print("Token stream : ✅ CREATED")

Building Validation Token Stream
Documents processed : 2,501
Documents skipped   : 0
Tokens collected    : 500,000

Token stream : ✅ CREATED


In [13]:
# ============================================================
# Cell 12: Build Validation Sequences
# ============================================================

from torch.utils.data import Dataset

print("=" * 75)
print("Building Validation Sequences")
print("=" * 75)


class ValidationSequenceDataset(Dataset):

    def __init__(
        self,
        tokens,
        sequence_length,
    ):
        self.tokens = torch.tensor(
            tokens,
            dtype=torch.long,
        )

        self.sequence_length = (
            sequence_length
        )

        # Need one additional token because
        # labels are shifted by one position.
        self.num_sequences = (
            len(self.tokens)
            // sequence_length
        )

    def __len__(self):
        return self.num_sequences

    def __getitem__(self, index):

        start = (
            index
            * self.sequence_length
        )

        end = (
            start
            + self.sequence_length
            + 1
        )

        chunk = self.tokens[
            start:end
        ]

        if len(chunk) < (
            self.sequence_length + 1
        ):
            raise IndexError(
                "Incomplete validation sequence."
            )

        input_ids = chunk[:-1]
        labels = chunk[1:]

        return input_ids, labels


validation_sequence_dataset = (
    ValidationSequenceDataset(
        all_tokens,
        SEQUENCE_LENGTH,
    )
)


print(
    f"Sequence Length      : "
    f"{SEQUENCE_LENGTH}"
)

print(
    f"Validation sequences : "
    f"{len(validation_sequence_dataset):,}"
)

print(
    f"Evaluation tokens     : "
    f"{len(validation_sequence_dataset) * SEQUENCE_LENGTH:,}"
)


if len(validation_sequence_dataset) == 0:
    raise RuntimeError(
        "No validation sequences were created."
    )

print()
print("Sequence generation : ✅ PASSED")

Building Validation Sequences
Sequence Length      : 512
Validation sequences : 976
Evaluation tokens     : 499,712

Sequence generation : ✅ PASSED


In [14]:
# ============================================================
# Cell 13: Sequence Inspection
# ============================================================

input_ids, labels = (
    validation_sequence_dataset[0]
)

print("=" * 75)
print("Validation Sequence Inspection")
print("=" * 75)

print(
    f"Input shape  : "
    f"{tuple(input_ids.shape)}"
)

print(
    f"Label shape  : "
    f"{tuple(labels.shape)}"
)

print(
    f"Input dtype  : "
    f"{input_ids.dtype}"
)

print(
    f"Label dtype  : "
    f"{labels.dtype}"
)

print()
print(
    "First 20 input tokens:"
)

print(
    input_ids[:20].tolist()
)

print()
print(
    "First 20 target tokens:"
)

print(
    labels[:20].tolist()
)


if input_ids.shape[0] != SEQUENCE_LENGTH:
    raise RuntimeError(
        "Incorrect input sequence length."
    )

if labels.shape[0] != SEQUENCE_LENGTH:
    raise RuntimeError(
        "Incorrect label sequence length."
    )

print()
print("Sequence structure : ✅ PASSED")

Validation Sequence Inspection
Input shape  : (512,)
Label shape  : (512,)
Input dtype  : torch.int64
Label dtype  : torch.int64

First 20 input tokens:
[2, 2753, 17, 2753, 609, 221, 2298, 728, 239, 350, 15, 333, 3447, 15, 10346, 15, 539, 728, 298, 371]

First 20 target tokens:
[2753, 17, 2753, 609, 221, 2298, 728, 239, 350, 15, 333, 3447, 15, 10346, 15, 539, 728, 298, 371, 2540]

Sequence structure : ✅ PASSED


In [15]:
# ============================================================
# Cell 14: Validation DataLoader
# ============================================================

from torch.utils.data import DataLoader

VALIDATION_BATCH_SIZE = 8

validation_loader = DataLoader(
    validation_sequence_dataset,
    batch_size=VALIDATION_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)


print("=" * 75)
print("Validation DataLoader")
print("=" * 75)

print(
    f"Batch Size       : "
    f"{VALIDATION_BATCH_SIZE}"
)

print(
    f"Number of batches: "
    f"{len(validation_loader):,}"
)

print()
print("DataLoader : ✅ CREATED")

Validation DataLoader
Batch Size       : 8
Number of batches: 122

DataLoader : ✅ CREATED


In [16]:
# ============================================================
# Cell 15: Run Validation
# ============================================================

import time
import math

print("=" * 75)
print("Running Validation Evaluation")
print("=" * 75)

model.eval()

total_loss = 0.0
total_tokens = 0

top1_correct = 0
top5_correct = 0
top10_correct = 0

evaluation_start = time.time()


with torch.no_grad():

    for batch_index, (
        input_ids,
        labels,
    ) in enumerate(validation_loader):

        input_ids = input_ids.to(
            DEVICE,
            non_blocking=True,
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True,
        )

        output = model(
            input_ids=input_ids,
            labels=labels,
        )


        # ----------------------------------------------------
        # Handle MyGPT2 output
        # ----------------------------------------------------

        if isinstance(output, tuple):

            if len(output) != 2:
                raise RuntimeError(
                    "Expected model output "
                    "format: (logits, loss)."
                )

            logits, loss = output

        else:

            if not hasattr(output, "loss"):
                raise RuntimeError(
                    "Model output has no loss."
                )

            logits = output.logits
            loss = output.loss


        if loss is None:
            raise RuntimeError(
                "Model returned None loss."
            )

        if not torch.isfinite(loss):
            raise RuntimeError(
                "Validation loss is NaN or infinite."
            )


        # ----------------------------------------------------
        # Loss
        # ----------------------------------------------------

        batch_token_count = (
            labels.numel()
        )

        total_loss += (
            loss.item()
            * batch_token_count
        )

        total_tokens += (
            batch_token_count
        )


        # ----------------------------------------------------
        # Accuracy
        # ----------------------------------------------------

        predictions = logits.argmax(
            dim=-1
        )

        top1_correct += (
            (predictions == labels)
            .sum()
            .item()
        )


        # ----------------------------------------------------
        # Top-K
        # ----------------------------------------------------

        flat_logits = logits.reshape(
            -1,
            logits.size(-1),
        )

        flat_labels = labels.reshape(
            -1
        )


        _, top5_predictions = (
            torch.topk(
                flat_logits,
                k=5,
                dim=-1,
            )
        )

        _, top10_predictions = (
            torch.topk(
                flat_logits,
                k=10,
                dim=-1,
            )
        )


        top5_correct += (
            (
                top5_predictions
                == flat_labels.unsqueeze(1)
            )
            .any(dim=1)
            .sum()
            .item()
        )


        top10_correct += (
            (
                top10_predictions
                == flat_labels.unsqueeze(1)
            )
            .any(dim=1)
            .sum()
            .item()
        )


        if (
            batch_index == 0
            or (batch_index + 1) % 10 == 0
        ):

            batch_accuracy = (
                predictions == labels
            ).float().mean().item()

            print(
                f"Batch {batch_index + 1:>5} | "
                f"Loss {loss.item():.6f} | "
                f"Accuracy "
                f"{batch_accuracy * 100:.2f}%"
            )


evaluation_time = (
    time.time()
    - evaluation_start
)


if total_tokens == 0:
    raise RuntimeError(
        "No validation tokens were evaluated."
    )


validation_loss = (
    total_loss
    / total_tokens
)


top1_accuracy = (
    top1_correct
    / total_tokens
)

top5_accuracy = (
    top5_correct
    / total_tokens
)

top10_accuracy = (
    top10_correct
    / total_tokens
)


validation_perplexity = math.exp(
    validation_loss
)


print()
print("=" * 75)
print("Validation Completed")
print("=" * 75)

print(
    f"Validation Loss     : "
    f"{validation_loss:.6f}"
)

print(
    f"Token Accuracy      : "
    f"{top1_accuracy * 100:.4f}%"
)

print(
    f"Top-5 Accuracy      : "
    f"{top5_accuracy * 100:.4f}%"
)

print(
    f"Top-10 Accuracy     : "
    f"{top10_accuracy * 100:.4f}%"
)

print(
    f"Tokens Evaluated    : "
    f"{total_tokens:,}"
)

print(
    f"Validation Batches  : "
    f"{len(validation_loader):,}"
)

print(
    f"Evaluation Time     : "
    f"{evaluation_time:.2f}s"
)

print("=" * 75)

Running Validation Evaluation
Batch     1 | Loss 3.165494 | Accuracy 39.09%
Batch    10 | Loss 2.092060 | Accuracy 54.76%
Batch    20 | Loss 3.411568 | Accuracy 35.79%
Batch    30 | Loss 3.829731 | Accuracy 33.35%
Batch    40 | Loss 3.283198 | Accuracy 37.84%
Batch    50 | Loss 3.997594 | Accuracy 31.54%
Batch    60 | Loss 3.941130 | Accuracy 31.88%
Batch    70 | Loss 3.914388 | Accuracy 31.81%
Batch    80 | Loss 3.380822 | Accuracy 38.21%
Batch    90 | Loss 4.180864 | Accuracy 30.93%
Batch   100 | Loss 2.744125 | Accuracy 44.19%
Batch   110 | Loss 2.814360 | Accuracy 43.51%
Batch   120 | Loss 4.179948 | Accuracy 32.32%

Validation Completed
Validation Loss     : 3.531280
Token Accuracy      : 36.6847%
Top-5 Accuracy      : 62.7223%
Top-10 Accuracy     : 70.7670%
Tokens Evaluated    : 499,712
Validation Batches  : 122
Evaluation Time     : 18.60s


In [17]:
# ============================================================
# Cell 16: Training vs Validation
# ============================================================

if training_loss is not None:

    training_loss_float = float(
        training_loss
    )

    training_perplexity = math.exp(
        training_loss_float
    )

    loss_gap = (
        validation_loss
        - training_loss_float
    )


    print("=" * 75)
    print("Training vs Validation")
    print("=" * 75)

    print(
        f"Training Loss       : "
        f"{training_loss_float:.6f}"
    )

    print(
        f"Validation Loss     : "
        f"{validation_loss:.6f}"
    )

    print(
        f"Loss Gap            : "
        f"{loss_gap:+.6f}"
    )

    print(
        f"Training Perplexity : "
        f"{training_perplexity:.4f}"
    )

    print(
        f"Validation Perplexity: "
        f"{validation_perplexity:.4f}"
    )


    if validation_loss <= (
        training_loss_float * 1.10
    ):

        generalization = "GOOD"

    elif validation_loss <= (
        training_loss_float * 1.25
    ):

        generalization = "ACCEPTABLE"

    else:

        generalization = "POSSIBLE OVERFITTING"


    print()
    print(
        f"Generalization     : "
        f"{generalization}"
    )

    print("=" * 75)

else:

    print(
        "Training loss is unavailable "
        "in the checkpoint."
    )

Training vs Validation
Training Loss       : 5.986759
Validation Loss     : 3.531280
Loss Gap            : -2.455479
Training Perplexity : 398.1223
Validation Perplexity: 34.1677

Generalization     : GOOD


In [18]:
# ============================================================
# Cell 17: Sample Token Predictions
# ============================================================

print("=" * 75)
print("Sample Validation Predictions")
print("=" * 75)


model.eval()

sample_input, sample_labels = (
    validation_sequence_dataset[0]
)

sample_input = sample_input.unsqueeze(0).to(
    DEVICE
)

sample_labels = sample_labels.to(
    DEVICE
)


with torch.no_grad():

    output = model(
        input_ids=sample_input,
        labels=sample_labels.unsqueeze(0),
    )

    if isinstance(output, tuple):

        sample_logits, _ = output

    else:

        sample_logits = output.logits


sample_predictions = (
    sample_logits
    .argmax(dim=-1)
    .squeeze(0)
)


print(
    "First 30 token predictions:"
)

print()

for i in range(30):

    predicted = (
        sample_predictions[i]
        .item()
    )

    actual = (
        sample_labels[i]
        .item()
    )

    mark = (
        "✓"
        if predicted == actual
        else "✗"
    )

    print(
        f"{i:>4} | "
        f"Predicted: {predicted:>6} | "
        f"Actual: {actual:>6} | "
        f"{mark}"
    )

Sample Validation Predictions
First 30 token predictions:

   0 | Predicted:    296 | Actual:   2753 | ✗
   1 | Predicted:    298 | Actual:     17 | ✗
   2 | Predicted:    180 | Actual:   2753 | ✗
   3 | Predicted:    298 | Actual:    609 | ✗
   4 | Predicted:    219 | Actual:    221 | ✗
   5 | Predicted:    549 | Actual:   2298 | ✗
   6 | Predicted:   7078 | Actual:    728 | ✗
   7 | Predicted:    251 | Actual:    239 | ✗
   8 | Predicted:    221 | Actual:    350 | ✗
   9 | Predicted:     15 | Actual:     15 | ✓
  10 | Predicted:    498 | Actual:    333 | ✗
  11 | Predicted:     44 | Actual:   3447 | ✗
  12 | Predicted:     15 | Actual:     15 | ✓
  13 | Predicted:    290 | Actual:  10346 | ✗
  14 | Predicted:      5 | Actual:     15 | ✗
  15 | Predicted:    311 | Actual:    539 | ✗
  16 | Predicted:    728 | Actual:    728 | ✓
  17 | Predicted:    298 | Actual:    298 | ✓
  18 | Predicted:    371 | Actual:    371 | ✓
  19 | Predicted:   1711 | Actual:   2540 | ✗
  20 | Predicted:    

In [19]:
# ============================================================
# Cell 18: Decode Validation Example
# ============================================================



print("=" * 75)
print("Validation Text Example")
print("=" * 75)


example_token_ids = (
    validation_sequence_dataset.tokens[
        :SEQUENCE_LENGTH
    ].tolist()
)


try:

    decoded_text = tokenizer.decode(
        example_token_ids
    )

except Exception:

    decoded_text = str(
        example_token_ids[:100]
    )


print()
print(decoded_text[:2000])

print()
print("=" * 75)

Validation Text Example

 Spot. Spot saw the shiny car and said, "Wow, Kitty, your car is so bright and clean!" Kitty smiled and replied, "Thank you, Spot. I polish it every day."

After playing with the car, Kitty and Spot felt thirsty. They found a small pond with clear water. They drank the water and felt very happy. They played together all day and became best friends. Once upon a time, in a big forest, there lived a rhinoceros named Roxy. Roxy loved to climb. She climbed trees, rocks, and hills. One day, Roxy found an icy hill. She had never seen anything like it before. It was shiny and cold, and she wanted to climb it.

Roxy tried to climb the icy hill, but it was very slippery. She tried again and again, but she kept falling down. Roxy was sad. She wanted to climb the icy hill so much. Then, she saw a little bird named Billy. Billy saw that Roxy was sad and asked, "Why are you sad, Roxy?"

Roxy told Billy about the icy hill and how she couldn't climb it. Billy said, "I have an 

In [20]:
# ============================================================
# Cell 19: Final Evaluation Summary
# ============================================================

print()
print("=" * 75)
print("MyGPT2 Validation Evaluation Summary")
print("=" * 75)

print(
    f"Checkpoint Step       : "
    f"{checkpoint_step:,}"
)

print(
    f"Model Parameters      : "
    f"{total_parameters:,}"
)

print(
    f"Validation Documents  : "
    f"{len(validation_dataset):,}"
)

print(
    f"Documents Evaluated   : "
    f"{documents_processed:,}"
)

print(
    f"Documents Skipped     : "
    f"{documents_skipped:,}"
)

print(
    f"Validation Sequences  : "
    f"{len(validation_sequence_dataset):,}"
)

print(
    f"Tokens Evaluated      : "
    f"{total_tokens:,}"
)

print()
print(
    f"Training Loss         : "
    f"{training_loss_float:.6f}"
    if training_loss is not None
    else "Training Loss         : N/A"
)

print(
    f"Validation Loss       : "
    f"{validation_loss:.6f}"
)

print(
    f"Training Perplexity   : "
    f"{training_perplexity:.4f}"
    if training_loss is not None
    else "Training Perplexity   : N/A"
)

print(
    f"Validation Perplexity : "
    f"{validation_perplexity:.4f}"
)

print()
print(
    f"Top-1 Accuracy        : "
    f"{top1_accuracy * 100:.4f}%"
)

print(
    f"Top-5 Accuracy        : "
    f"{top5_accuracy * 100:.4f}%"
)

print(
    f"Top-10 Accuracy       : "
    f"{top10_accuracy * 100:.4f}%"
)

print()
print(
    f"Evaluation Time       : "
    f"{evaluation_time:.2f}s"
)

print()
print(
    f"Generalization       : "
    f"{generalization}"
)

print()
print(
    "Validation Evaluation : ✅ COMPLETED"
)

print("=" * 75)


MyGPT2 Validation Evaluation Summary
Checkpoint Step       : 20,000
Model Parameters      : 110,025,216
Validation Documents  : 21,990
Documents Evaluated   : 2,501
Documents Skipped     : 0
Validation Sequences  : 976
Tokens Evaluated      : 499,712

Training Loss         : 5.986759
Validation Loss       : 3.531280
Training Perplexity   : 398.1223
Validation Perplexity : 34.1677

Top-1 Accuracy        : 36.6847%
Top-5 Accuracy        : 62.7223%
Top-10 Accuracy       : 70.7670%

Evaluation Time       : 18.60s

Generalization       : GOOD

Validation Evaluation : ✅ COMPLETED


In [22]:
# ============================================================
# Cell 20: Save Evaluation Results
# ============================================================

import json

RESULTS_DIR = (
    PROJECT_ROOT
    / "evaluation"
    / "results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


results = {
    "checkpoint": {
        "path": str(CHECKPOINT_PATH),
        "version": checkpoint.get(
            "checkpoint_version"
        ),
        "epoch": checkpoint_epoch,
        "global_step": checkpoint_step,
    },

    "model": {
        "parameters": total_parameters,
        "trainable_parameters":
            trainable_parameters,
        "vocab_size": VOCAB_SIZE,
        "sequence_length":
            SEQUENCE_LENGTH,
        "hidden_size": HIDDEN_SIZE,
        "num_layers": NUM_LAYERS,
        "num_heads": NUM_HEADS,
        "intermediate_size":
            INTERMEDIATE_SIZE,
    },

    "dataset": {
        "name":
            "roneneldan/TinyStories",
        "split":
            "validation",
        "documents_total":
            len(validation_dataset),
        "documents_processed":
            documents_processed,
        "documents_skipped":
            documents_skipped,
        "sequences":
            len(validation_sequence_dataset),
        "tokens_evaluated":
            total_tokens,
    },

    "metrics": {
        "training_loss":
            float(training_loss)
            if training_loss is not None
            else None,

        "validation_loss":
            float(validation_loss),

        "training_perplexity":
            float(training_perplexity)
            if training_loss is not None
            else None,

        "validation_perplexity":
            float(validation_perplexity),

        "top1_accuracy":
            float(top1_accuracy),

        "top5_accuracy":
            float(top5_accuracy),

        "top10_accuracy":
            float(top10_accuracy),
    },

    "evaluation": {
        "device": str(DEVICE),
        "evaluation_time_seconds":
            float(evaluation_time),
    },
}


RESULTS_PATH = (
    RESULTS_DIR
    / "validation_step_10000.json"
)


with open(
    RESULTS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        results,
        f,
        indent=4,
    )


print("=" * 75)
print("Evaluation Results Saved")
print("=" * 75)

print(
    f"Results : {RESULTS_PATH}"
)

print()
print("Results file : ✅ SAVED")

Evaluation Results Saved
Results : D:\Gpt2_v01\evaluation\results\validation_step_10000.json

Results file : ✅ SAVED
